# 2 — Flyback Converter Controller Design

> **Goal.** Size a Type-III compensator for the flyback respecting
> the RHP zero, discretize it via Tustin, and **prove** it works by
> running a switched closed-loop simulation with a $v_{ref}$ step.

**Prerequisites**

- Flyback modeling notebook (`01_flyback_modeling.ipynb`).
- The buck / boost / buck-boost controller notebooks for context.

The recipe is the same K-factor Type-III approach. What differs is
$f_{z,RHP}$ (and therefore $f_c$) being HIGHER than the buck-boost's
at the same $D$ — so the flyback's closed loop can be faster.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from flyback_model import (
    FlybackParams, control_to_output_tf, operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

params = FlybackParams()
print(operating_point_report(params))


## 1. Bandwidth target

$f_{z,RHP}$ for the default flyback ≈ 12.7 kHz. Target $f_c = f_z/5
\\approx 2.5$ kHz, PM = 60°. That's 30 % faster than the buck-boost
at the same operating point.


In [ ]:
Gvd = control_to_output_tf(params)
V_ramp = 5.0
plant = signal.TransferFunction(np.array(Gvd.num)/V_ramp, np.array(Gvd.den))

f_c_target = params.f_z_rhp / 5.0
pm_target = 60.0
print(f"RHP zero:   f_z = {params.f_z_rhp:7.0f} Hz")
print(f"Target:     f_c = {f_c_target:.0f} Hz, PM = {pm_target}°")


## 2. K-factor Type-III design

Same algorithm as buck-boost (including the phase-unwrap fix for
non-minimum-phase plants).


In [ ]:
def design_type3_kfactor(plant, f_c, pm_target):
    omega_c = 2 * np.pi * f_c
    _, _, ph_plant = signal.bode(plant, w=[omega_c])
    # scipy wraps phase to [-180°, +180°]; unfold past -180° for
    # non-minimum-phase plants (RHP zero past the LC double pole).
    ph_at_fc = ph_plant[0]
    if ph_at_fc > 0:
        ph_at_fc -= 360.0
    phi_lead = pm_target - 90.0 - ph_at_fc
    phi_lead = float(np.clip(phi_lead, 10.0, 175.0))
    phi_pair = phi_lead / 2
    k = np.tan(np.deg2rad(phi_pair/2 + 45.0))**2
    omega_z = omega_c / np.sqrt(k)
    omega_p = omega_c * np.sqrt(k)
    num0 = np.polymul([1.0, omega_z], [1.0, omega_z])
    den0 = np.polymul([1.0, 0.0], np.polymul([1.0, omega_p], [1.0, omega_p]))
    open0 = signal.TransferFunction(
        np.polymul(num0, plant.num), np.polymul(den0, plant.den)
    )
    _, mag0, _ = signal.bode(open0, w=[omega_c])
    K = 10.0 ** (-mag0[0] / 20)
    return signal.TransferFunction(K * num0, den0), omega_z, omega_p, float(K)


Gc, omega_z, omega_p, K_dc = design_type3_kfactor(plant, f_c_target, pm_target)
print(f"Designed compensator:")
print(f"  zeros at  f_z = {omega_z/(2*np.pi):8.1f} Hz  (double)")
print(f"  HF poles  f_p = {omega_p/(2*np.pi):8.1f} Hz  (double)")
print(f"  DC gain K     = {K_dc:.4g}")


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f
T_open = signal.TransferFunction(np.polymul(Gc.num, plant.num),
                                   np.polymul(Gc.den, plant.den))
_, mag_T, ph_T = signal.bode(T_open, w=w)
_, mag_p, ph_p = signal.bode(plant, w=w)
idx_cross = np.argmin(np.abs(mag_T))
f_cross = f[idx_cross]
pm = 180 + ph_T[idx_cross]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_mag.semilogx(f, mag_p, "C0--", alpha=0.5, label="Plant + $k_{PWM}$")
ax_mag.semilogx(f, mag_T, "C3", linewidth=2, label="Loop gain $T(s)$")
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_mag.axvline(f_cross, color="g", linestyle=":", alpha=0.5,
               label=f"$f_c$ ≈ {f_cross:.0f} Hz")
ax_mag.axvline(params.f_z_rhp, color="C3", linestyle=":", alpha=0.4,
               label=r"$f_{z,RHP}$")
ax_ph.semilogx(f, ph_p, "C0--", alpha=0.5)
ax_ph.semilogx(f, ph_T, "C3", linewidth=2)
ax_ph.axhline(-180, color="k", linestyle=":", alpha=0.3)
ax_ph.axvline(f_cross, color="g", linestyle=":", alpha=0.5)
ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.legend(loc="best", fontsize=8)
ax_mag.set_title(f"Compensated flyback loop: $f_c$ = {f_cross:.0f} Hz, "
                 f"PM = {pm:.1f}°")
plt.tight_layout()
plt.show()

print(f"Target:    f_c = {f_c_target:.0f} Hz, PM = {pm_target}°")
print(f"Achieved:  f_c = {f_cross:.0f} Hz, PM = {pm:.1f}°")


## 3. Discretization

Tustin / bilinear at $T_s = 1/f_{sw}$.


In [ ]:
T_s = 1.0 / params.f_sw
Gc_d_num, Gc_d_den, _ = signal.cont2discrete(
    (Gc.num, Gc.den), dt=T_s, method="bilinear"
)
b = np.asarray(Gc_d_num).flatten() / Gc_d_den[0]
a = np.asarray(Gc_d_den) / Gc_d_den[0]
print(f"Sample period T_s = {T_s*1e6:.3f} µs")
print()
print("Discrete-time recurrence (a[0] = 1):")
for i, bi in enumerate(b): print(f"  b[{i}] = {bi:+.6f}")
for i, ai in enumerate(a): print(f"  a[{i}] = {ai:+.6f}")


## 4. Switched-model closed-loop simulation

Pure-Python forward-Euler flyback + digital compensator running once
per switching period (sample-and-hold).

The switched model:

```
ON  (S closed, D off):
    L_m · di_Lm/dt = V_g
    C   · dv_o/dt  = -v_o / R

OFF (S open, D on, in CCM with i_Lm > 0):
    L_m · di_Lm/dt = -v_o / n
    C   · dv_o/dt  = i_Lm / n - v_o / R
```

Warm-start at the operating point (with $i_{Lm}$ at the
beginning-of-ON valley value — same fix as the buck-boost notebook).


In [ ]:
def simulate_closed_loop_flyback(
    params,
    b: np.ndarray, a: np.ndarray,
    *,
    t_end: float = 20e-3,
    t_step: float = 5e-3,
    v_ref_initial: float = 12.0,
    v_ref_final: float = 13.0,
    V_ramp: float = 5.0,
    samples_per_period: int = 200,
    warm_start: bool = True,
):
    '''Forward-Euler switched flyback + digital compensator.

    States are primary magnetizing current i_Lm and secondary cap
    voltage v_o. The compensator runs once per switching period.

    warm_start=True initializes both plant states and compensator
    state at the operating point for v_ref_initial. Uses VALLEY i_Lm
    (= average - ripple/2) to avoid the half-period charge offset
    that plagued the buck-boost cold start.
    '''
    T_s = 1.0 / params.f_sw
    dt_sim = T_s / samples_per_period
    n_steps = int(t_end / dt_sim) + 1
    n_turn = params.n

    n_state = len(a) - 1
    state = np.zeros(n_state)

    if warm_start:
        # D from V_o = n·V_g·D/(1-D):
        D_init = v_ref_initial / (v_ref_initial + n_turn * params.V_g)
        I_Lm_avg = v_ref_initial * n_turn / (params.R * (1.0 - D_init))
        T_s_period = 1.0 / params.f_sw
        # Primary magnetizing ripple over ON interval: di/dt = V_g/L
        delta_i_pp = params.V_g * D_init * T_s_period / params.L_m
        i_Lm = I_Lm_avg - delta_i_pp / 2.0
        v_o = v_ref_initial
        duty = D_init
        v_c_ss = duty * V_ramp
        for k in range(n_state):
            state[k] = -np.sum(a[k+1:]) * v_c_ss
    else:
        i_Lm = 0.0
        v_o = 0.0
        duty = 0.5

    record_every = max(1, samples_per_period // 50)
    n_rec = n_steps // record_every + 1
    t_hist = np.zeros(n_rec)
    v_o_hist = np.zeros(n_rec)
    i_Lm_hist = np.zeros(n_rec)
    duty_hist = np.zeros(n_rec)
    v_ref_hist = np.zeros(n_rec)
    rec_idx = 0

    for i in range(n_steps):
        t = i * dt_sim
        v_ref = v_ref_initial if t < t_step else v_ref_final

        cycle_pos_int = i % samples_per_period
        if cycle_pos_int == 0:
            err = v_ref - v_o
            v_c = b[0] * err + state[0]
            new_state = np.zeros_like(state)
            for j in range(n_state - 1):
                new_state[j] = b[j+1] * err - a[j+1] * v_c + state[j+1]
            new_state[n_state - 1] = b[n_state] * err - a[n_state] * v_c
            state = new_state
            duty = float(np.clip(v_c / V_ramp, 0.05, 0.92))

        switch_on = (cycle_pos_int / samples_per_period) < duty

        # Switched flyback ODE
        if switch_on:
            v_Lm = params.V_g
            i_C  = -v_o / params.R
        else:
            if i_Lm > 0:
                v_Lm = -v_o / n_turn
                i_C  = i_Lm / n_turn - v_o / params.R
            else:
                v_Lm = 0.0     # diode off (DCM)
                i_C  = -v_o / params.R
        i_Lm += (v_Lm / params.L_m) * dt_sim
        i_Lm = max(i_Lm, 0.0)
        v_o += (i_C / params.C) * dt_sim

        if i % record_every == 0 and rec_idx < n_rec:
            t_hist[rec_idx] = t
            v_o_hist[rec_idx] = v_o
            i_Lm_hist[rec_idx] = i_Lm
            duty_hist[rec_idx] = duty
            v_ref_hist[rec_idx] = v_ref
            rec_idx += 1

    return {
        "t": t_hist[:rec_idx], "v_o": v_o_hist[:rec_idx],
        "i_Lm": i_Lm_hist[:rec_idx], "duty": duty_hist[:rec_idx],
        "v_ref": v_ref_hist[:rec_idx],
    }


In [ ]:
sim = simulate_closed_loop_flyback(
    params, b=b, a=a,
    t_end=20e-3, t_step=5e-3,
    v_ref_initial=12.0, v_ref_final=13.0,
    V_ramp=V_ramp, warm_start=True,
)

print(f"Simulated {len(sim['t'])} samples over {sim['t'][-1]*1e3:.2f} ms")
pre_mask  = (sim['t'] > 4.0e-3) & (sim['t'] < 5.0e-3)
post_mask = sim['t'] > 17.0e-3
print()
print(f"Pre-step  V_o (mean 4-5 ms):    {np.mean(sim['v_o'][pre_mask]):.4f} V "
      f"(target 12.0)")
print(f"Post-step V_o (mean 17-20 ms):  {np.mean(sim['v_o'][post_mask]):.4f} V "
      f"(target 13.0)")
D_pre = 12.0 / (12.0 + params.n * params.V_g)
D_post = 13.0 / (13.0 + params.n * params.V_g)
print(f"Pre-step duty:  {np.mean(sim['duty'][pre_mask]):.4f} (expect {D_pre:.4f})")
print(f"Post-step duty: {np.mean(sim['duty'][post_mask]):.4f} (expect {D_post:.4f})")


In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axs[0].plot(sim['t']*1e3, sim['v_o'], 'C0', linewidth=0.8, label="$v_o$ (switched)")
axs[0].plot(sim['t']*1e3, sim['v_ref'], 'C3--', linewidth=2, label="$v_{ref}$")
axs[0].axvline(5.0, color="k", linestyle=":", alpha=0.4, label="step")
axs[0].set_ylabel("Output voltage [V]")
axs[0].set_title("Closed-loop flyback (warm-start at OP): step $v_{ref}$ "
                 "12 V → 13 V at $t$ = 5 ms")
axs[0].legend(loc="lower right")

axs[1].plot(sim['t']*1e3, sim['i_Lm'], 'C1', linewidth=0.8)
axs[1].axvline(5.0, color="k", linestyle=":", alpha=0.4)
I_Lm_pre = 12.0 * params.n / (params.R * (1 - D_pre))
I_Lm_post = 13.0 * params.n / (params.R * (1 - D_post))
axs[1].axhline(I_Lm_pre, color="k", linestyle=":", alpha=0.3,
               label=f"pre-step $I_{{L_m}}$ = {I_Lm_pre:.2f} A")
axs[1].axhline(I_Lm_post, color="r", linestyle=":", alpha=0.3,
               label=f"post-step $I_{{L_m}}$ = {I_Lm_post:.2f} A")
axs[1].set_ylabel("Primary mag current [A]")
axs[1].legend(loc="lower right")

axs[2].plot(sim['t']*1e3, sim['duty'], 'C2', linewidth=1.0)
axs[2].axvline(5.0, color="k", linestyle=":", alpha=0.4)
axs[2].axhline(D_pre, color="k", linestyle=":", alpha=0.3,
               label=f"pre-step D = {D_pre:.3f}")
axs[2].axhline(D_post, color="r", linestyle=":", alpha=0.3,
               label=f"post-step D = {D_post:.3f}")
axs[2].set_ylabel("Duty cycle")
axs[2].legend(loc="lower right")

axs[3].plot(sim['t']*1e3, sim['v_ref'] - sim['v_o'], 'C4', linewidth=0.8)
axs[3].axvline(5.0, color="k", linestyle=":", alpha=0.4)
axs[3].axhline(0, color="k", linestyle=":", alpha=0.3)
axs[3].set_ylabel("Tracking error\n$v_{ref} - v_o$ [V]")
axs[3].set_xlabel("Time [ms]")

plt.tight_layout()
plt.show()


In [ ]:
mask_after = sim['t'] > 5.0e-3
t_after = sim['t'][mask_after] - 5.0e-3
v_o_after = sim['v_o'][mask_after]

overshoot_pct = (np.max(v_o_after) - 13.0) / (13.0 - 12.0) * 100
dip_amount = 12.0 - np.min(v_o_after)
settled = np.abs(v_o_after - 13.0) < 0.02 * (13.0 - 12.0)
unsettled = np.where(~settled)[0]
settling_ms = t_after[
    min(unsettled[-1] + 1, len(t_after) - 1) if len(unsettled) else 0
] * 1e3
v_o_10 = 12.0 + 0.1
v_o_90 = 12.0 + 0.9
rise_start = np.argmax(v_o_after >= v_o_10)
rise_end = np.argmax(v_o_after >= v_o_90)
rise_time_ms = (t_after[rise_end] - t_after[rise_start]) * 1e3

ss_error = 13.0 - np.mean(sim['v_o'][sim['t'] > 17e-3])

print("Closed-loop step-response metrics ($v_{ref}$: 12 → 13 V)")
print(f"  Initial dip (RHP zero)     = {dip_amount * 1e3:7.1f} mV below pre-step")
print(f"  Rise time (10% → 90%)      = {rise_time_ms:7.3f} ms")
print(f"  Peak overshoot             = {overshoot_pct:7.2f} %")
print(f"  Settling time (±2 %)       = {settling_ms:7.3f} ms")
print(f"  Steady-state error         = {ss_error*1e3:+7.2f} mV "
      f"({ss_error / 13.0 * 100:+.3f} %)")
print()
if abs(ss_error) < 0.15 and overshoot_pct < 50 and settling_ms < 30.0:
    print("✅  Closed-loop flyback controller PROVEN:")
    print(f"    • SS error    = {ss_error*1e3:.1f} mV ({ss_error/13.0*100:.2f} %)")
    print(f"    • Overshoot   = {overshoot_pct:.1f} %")
    print(f"    • Settling    = {settling_ms:.1f} ms")
    print(f"    • RHP dip     = {dip_amount*1e3:.1f} mV")
    print()
    print(f"    Faster than buck-boost (~25 ms settling at same component values)")
    print(f"    because the transformer's $1/n^2$ reflection pushes $f_{{z,RHP}}$ up,")
    print(f"    giving the loop more bandwidth headroom.")
else:
    print("⚠️   Closed-loop response off-target — revisit f_c or PM.")


## 5. Summary

You designed a Type-III compensator for the flyback respecting the
RHP zero, discretized via Tustin, and confirmed it tracks $v_{ref}$
steps on the actual switched waveform. The flyback's loop is **faster**
than the buck-boost's at the same component values — that's the
transformer's $1/n^2$ reflection giving the engineer extra bandwidth
headroom.

**Suggested exercises**

1. Push $n$ to 1.0 (no step-up, no step-down — the flyback is
   essentially a buck-boost). How much does $f_c$ drop?
2. Push $n$ to 0.25 (4:1 step-down). How much faster does the loop
   become? At what point does the LC pole's lightly-damped peak
   become unmanageable?
3. Design a flyback for a 12 V → 5 V phone-charger use case. What
   $n$ and $L_m$ would you pick? Plot $f_{z,RHP}$ vs $V_o$ at fixed
   $V_g$ and $P_{out}$.
